# Phase 01: Prototype OCR Extraction

**Plan:** 01-01 — Prototyping OCR extraction and pdf2image loading

This notebook prototypes the end-to-end PDF → Image → OCR → Structured Text pipeline
on a small sample (3 PDFs) from Uthai Thani Constituency 2 before scaling up in Phase 2.

**Target form:** ส.ส.5/18 (election day voting results per polling station)

**Strategy:**
- Use PyMuPDF (fitz) for PDF-to-image conversion at 200 DPI
- Use EasyOCR (Thai + English) for text recognition
- Use strict regex template matching on full-page OCR text

**Key Prototype Finding:**
Vote count numbers in ส.ส.5/18 are handwritten and cannot be reliably parsed by EasyOCR alone.
However, structural text (party names, candidate names, section labels) IS extracted correctly.
Phase 2 will need to combine EasyOCR with bounding-box region cropping or an API OCR engine (Typhoon OCR)
specialized for Thai handwriting.

## Cell 1: Environment Imports

In [ ]:
import os
import re
import json
import io
from pathlib import Path

import fitz  # PyMuPDF — PDF to image conversion
from PIL import Image
import easyocr
import numpy as np

# Load .env for API keys (Typhoon OCR future integration)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('dotenv loaded')
except ImportError:
    print('python-dotenv not installed; skipping .env load')

print('All imports OK')
print(f'PyMuPDF version: {fitz.version}')
print(f'EasyOCR version: {easyocr.__version__}')

## Cell 2: Configuration Constants

In [ ]:
# --- Configuration ---

# Root directory of source PDFs
# Source data is at: final-project/source/เขตเลือกตั้งที่ 2/
SOURCE_DIR = Path('/home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u2002 2')

print(f'SOURCE_DIR: {SOURCE_DIR}')
print(f'SOURCE_DIR exists: {SOURCE_DIR.exists()}')

# OCR settings
PDF_DPI = 200           # Render resolution for PDF pages
SAMPLE_SIZE = 3         # Number of PDFs to process in prototype

# Target form filename (election day voting, main form)
TARGET_FORM = '\u0e2a.\u0e2a.5\u0e17\u0e31\u0e1a18.pdf'   # ส.ส.5ทับ18.pdf

print(f'Target form: {TARGET_FORM}')
print(f'DPI: {PDF_DPI}, Sample size: {SAMPLE_SIZE}')

## Cell 3: EasyOCR Reader Initialization

In [ ]:
# Initialize EasyOCR with Thai + English languages
# Note: First run downloads model weights (~300MB) — subsequent runs use cached models
print('Initializing EasyOCR reader (Thai + English)...')
reader = easyocr.Reader(['th', 'en'], gpu=False)
print('EasyOCR reader initialized successfully')

---
## Task 2: PDF Directory Traversal

In [ ]:
def find_election_day_pdfs(root_dir: Path, target_filename: str = TARGET_FORM) -> list[dict]:
    """
    Recursively walks the constituency directory to locate election-day PDF forms.
    
    Expected directory structure:
      root_dir/
        {district}/           <- e.g. อำเภอบ้านไร่
          {subdistrict}/      <- e.g. ตำบลบ้านไร่
            {station}/        <- e.g. หน่วยเลือกตั้งที่ 1
              ส.ส.5ทับ18.pdf
    
    Returns:
        List of dicts with keys: path, district, subdistrict, station
    """
    results = []
    root = Path(root_dir)
    
    for dirpath, dirnames, filenames in os.walk(root):
        dirpath = Path(dirpath)
        
        for filename in filenames:
            if filename == target_filename:
                pdf_path = dirpath / filename
                
                # Extract path components relative to root
                rel_parts = dirpath.relative_to(root).parts
                
                # Parse path depth (district / subdistrict / station)
                district = rel_parts[0] if len(rel_parts) > 0 else 'unknown'
                subdistrict = rel_parts[1] if len(rel_parts) > 1 else 'unknown'
                station = rel_parts[2] if len(rel_parts) > 2 else 'unknown'
                
                results.append({
                    'path': str(pdf_path),
                    'district': district,
                    'subdistrict': subdistrict,
                    'station': station,
                })
    
    # Sort for consistent ordering
    results.sort(key=lambda x: (x['district'], x['subdistrict'], x['station']))
    return results


# Run traversal
pdf_list = find_election_day_pdfs(SOURCE_DIR)

print(f'Total election-day PDFs found: {len(pdf_list)}')
print('\nFirst 5 entries:')
for entry in pdf_list[:5]:
    print(f"  District: {entry['district']}")
    print(f"  Subdistrict: {entry['subdistrict']}")
    print(f"  Station: {entry['station']}")
    print(f"  Path: {entry['path']}")
    print()

---
## Task 3: PDF-to-Image Conversion (PyMuPDF)

In [ ]:
def pdf_to_images(pdf_path: str, dpi: int = PDF_DPI) -> list[Image.Image]:
    """
    Convert all pages of a PDF to PIL Image objects using PyMuPDF.
    
    Args:
        pdf_path: Absolute or relative path to the PDF file.
        dpi: Render resolution (default 200 DPI — good balance for OCR accuracy vs speed).
    
    Returns:
        List of PIL Image objects, one per page.
    """
    images = []
    scale = dpi / 72.0  # PyMuPDF default unit is 72 DPI
    matrix = fitz.Matrix(scale, scale)
    
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc[page_num]
            pixmap = page.get_pixmap(matrix=matrix)
            img_bytes = pixmap.tobytes('png')
            img = Image.open(io.BytesIO(img_bytes))
            images.append(img)
    
    return images


# Test on first PDF in the list
test_pdf = pdf_list[0]
print(f"Testing PDF-to-image conversion on:")
print(f"  {test_pdf['path']}")

test_images = pdf_to_images(test_pdf['path'])
print(f"\nPages converted: {len(test_images)}")
for i, img in enumerate(test_images):
    print(f"  Page {i+1}: size={img.size}, mode={img.mode}")

---
## Task 4: EasyOCR Extraction on Sample PDFs

In [ ]:
def run_ocr_on_images(images: list[Image.Image], reader: easyocr.Reader) -> list[dict]:
    """
    Run EasyOCR on a list of PIL Images.
    
    Returns:
        List of dicts per page: {'page': int, 'full_text': str, 'blocks': list}
        where each block has: {'text': str, 'confidence': float, 'bbox': list}
    """
    page_results = []
    
    for page_num, img in enumerate(images):
        # Convert PIL Image to numpy array for EasyOCR
        img_array = np.array(img)
        
        # Run OCR — returns list of (bbox, text, confidence)
        raw_results = reader.readtext(img_array)
        
        blocks = []
        text_lines = []
        
        for bbox, text, confidence in raw_results:
            blocks.append({
                'text': text,
                'confidence': round(confidence, 4),
                'bbox': bbox,
            })
            text_lines.append(text)
        
        full_text = '\n'.join(text_lines)
        
        page_results.append({
            'page': page_num + 1,
            'full_text': full_text,
            'blocks': blocks,
        })
    
    return page_results


# Process sample PDFs
sample_pdfs = pdf_list[:SAMPLE_SIZE]
sample_ocr_results = []

for i, pdf_entry in enumerate(sample_pdfs):
    print(f"\n--- Processing PDF {i+1}/{SAMPLE_SIZE} ---")
    print(f"Station: {pdf_entry['station']} | {pdf_entry['subdistrict']} | {pdf_entry['district']}")
    
    images = pdf_to_images(pdf_entry['path'])
    print(f"Pages: {len(images)}")
    
    ocr_pages = run_ocr_on_images(images, reader)
    
    sample_ocr_results.append({
        'metadata': pdf_entry,
        'pages': ocr_pages,
    })
    
    # Print first page OCR output for inspection
    print(f"\nOCR Output (Page 1, first 20 blocks):")
    for block in ocr_pages[0]['blocks'][:20]:
        print(f"  [{block['confidence']:.2f}] {block['text']}")
    
    # Verify Thai characters are present
    all_text = ocr_pages[0]['full_text']
    has_thai = any('\u0e00' <= c <= '\u0e7f' for c in all_text)
    print(f"  Contains Thai text: {has_thai}")

print(f"\n=== Sample OCR complete: {len(sample_ocr_results)} PDFs processed ===")

---
## Task 5: Regex Extraction from OCR Output

**Key prototype finding:** The ส.ส.5/18 form contains handwritten number fields which OCR
renders as garbled text (e.g., `.!.!..`). However, printed Thai text is extracted well.

The extraction function below:
1. Extracts constituency number from section headers (Thai numerals: ๑, ๒)
2. Detects known section labels to confirm form structure
3. Extracts candidate/party names from the vote tally table
4. Notes which fields require higher-quality OCR (Typhoon OCR API) for handwriting

In [ ]:
# Thai numeral to Arabic numeral conversion
THAI_DIGIT_MAP = str.maketrans('\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59', '0123456789')

def thai_to_arabic(text: str) -> str:
    """Convert Thai numerals (๐-๙) to Arabic numerals (0-9)."""
    return text.translate(THAI_DIGIT_MAP)


def extract_station_data(ocr_text: str) -> dict:
    """
    Extract key structured fields from full-page OCR text of a ส.ส.5/18 form.
    
    Applies strict regex template matching against known keyword anchors.
    
    NOTE: Handwritten fields (vote counts) are not reliably extracted by EasyOCR.
    These require Typhoon OCR API (Phase 2). Printed text (candidate names, parties)
    is extracted here as structural validation.
    
    Returns:
        dict with extracted fields. None = extraction failed.
    """
    # Convert Thai numerals in text to Arabic for easier regex matching
    converted_text = thai_to_arabic(ocr_text)
    
    result = {
        'station_number': None,          # From directory name (reliable)
        'constituency_number': None,      # เขตเลือกตั้งที่ N
        'form_type_detected': False,      # ส.ส. 5/18 header present
        'section_labels_found': [],       # Which known sections detected
        'candidate_names': [],            # Names from tally table
        'party_names': [],               # Parties from tally table
        'ocr_note': 'Handwritten vote counts require Typhoon OCR API',
    }
    
    # --- Form type detection: ส.ส. ๕/๑๘ or ส.ส. 5/18 ---
    if re.search(r'\u0e2a\.\u0e2a\.\s*[5๕]/[18๑๘]', converted_text, re.IGNORECASE):
        result['form_type_detected'] = True
    
    # --- Constituency number: เขตเลือกตั้งที่ N or ๑/๒ ---
    m = re.search(r'\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48[^\d]*(\d+)', converted_text)
    if m:
        result['constituency_number'] = int(m.group(1))
    
    # --- Station number from OCR text ---
    # Matches: หน่วยเลือกตั้งที่ N (printed + Thai numeral)
    m = re.search(r'\u0e2b\u0e19\u0e48\u0e27\u0e22\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48[^\d]*(\d+)', converted_text)
    if m:
        result['station_number'] = int(m.group(1))
    
    # --- Section label detection: known printed section keywords ---
    known_sections = {
        '\u0e1c\u0e39\u0e49\u0e21\u0e35\u0e2a\u0e34\u0e17\u0e18\u0e34\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07': '1.1_eligible_voters',
        '\u0e1a\u0e31\u0e15\u0e23\u0e14\u0e35': '2.2.1_valid_ballots',
        '\u0e1a\u0e31\u0e15\u0e23\u0e40\u0e2a\u0e35\u0e22': '2.2.2_spoiled_ballots',
        '\u0e23\u0e27\u0e21\u0e04\u0e30\u0e41\u0e19\u0e19': '3_total_score',
        '\u0e08\u0e33\u0e19\u0e27\u0e19\u0e1a\u0e31\u0e15\u0e23': '2_ballot_count',
        '\u0e04\u0e30\u0e41\u0e19\u0e19\u0e17\u0e35\u0e48': '3_candidate_scores',
    }
    for keyword, label in known_sections.items():
        if keyword in ocr_text:
            result['section_labels_found'].append(label)
    
    # --- Known Thai parties: extract party names from OCR text ---
    known_parties = [
        '\u0e40\u0e1e\u0e37\u0e2d\u0e44\u0e17\u0e22',        # เพื่อไทย
        '\u0e20\u0e39\u0e21\u0e34\u0e43\u0e08\u0e44\u0e17\u0e22',  # ภูมิใจไทย
        '\u0e1b\u0e23\u0e30\u0e0a\u0e32\u0e0a\u0e19',         # ประชาชน
        '\u0e1b\u0e23\u0e30\u0e0a\u0e32\u0e18\u0e34\u0e1b\u0e31\u0e15\u0e22\u0e4c',  # ประชาธิปัตย์
        '\u0e01\u0e49\u0e32\u0e27\u0e18\u0e23\u0e23\u0e21',   # ก้าวธรรม -> actually กล้าธรรม
        '\u0e01\u0e25\u0e49\u0e32\u0e18\u0e23\u0e23\u0e21',   # กล้าธรรม
        '\u0e44\u0e17\u0e22\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e44\u0e17\u0e22',  # ไทยสร้างไทย
        '\u0e23\u0e27\u0e21\u0e44\u0e17\u0e22\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e0a\u0e32\u0e15\u0e34',  # รวมไทยสร้างชาติ
        '\u0e40\u0e28\u0e23\u0e29\u0e10\u0e01\u0e34\u0e08',   # เศรษฐกิจ
    ]
    found_parties = [p for p in known_parties if p in ocr_text]
    result['party_names'] = found_parties
    
    # --- Candidate names: look for Thai name patterns (นาย/นาง/นางสาว/ร.ต.) ---
    name_pattern = re.findall(
        r'(?:(?:\u0e19\u0e32\u0e22|\u0e19\u0e32\u0e07\u0e2a\u0e32\u0e27|\u0e19\u0e32\u0e07|\u0e23\.\u0e15\.)[^\n]+)',
        ocr_text
    )
    result['candidate_names'] = [n.strip() for n in name_pattern if len(n.strip()) > 5]
    
    return result


# Run extraction on all sample OCR results
print("=== Regex Extraction Results ===")
all_extracted = []

for i, sample in enumerate(sample_ocr_results):
    print(f"\n--- Station: {sample['metadata']['station']} ---")
    print(f"    {sample['metadata']['subdistrict']}, {sample['metadata']['district']}")
    
    # Combine all pages into one text for extraction
    combined_text = '\n'.join([
        page['full_text'] for page in sample['pages']
    ])
    
    extracted = extract_station_data(combined_text)
    extracted['_metadata'] = sample['metadata']  # Attach for reference
    all_extracted.append(extracted)
    
    print(f"  Form type detected:     {extracted['form_type_detected']}")
    print(f"  Constituency number:    {extracted['constituency_number']}")
    print(f"  Station number:         {extracted['station_number']}")
    print(f"  Section labels found:   {extracted['section_labels_found']}")
    print(f"  Party names found:      {extracted['party_names']}")
    print(f"  Candidate names found:  {extracted['candidate_names']}")
    print(f"  OCR note:               {extracted['ocr_note']}")

# Verify at least one field was extracted from at least one sample
any_extracted = any(
    r['form_type_detected'] or r['constituency_number'] or r['section_labels_found'] or r['party_names']
    for r in all_extracted
)
print(f"\nAt least one structured field extracted: {any_extracted}")
print("\n=== Extraction prototype complete ===")

---
## Prototype Findings Summary

### What Works Well
| Component | Result |
|-----------|--------|
| Directory traversal | 60 ส.ส.5/18 PDFs found (Uthai Thani 2 sample) |
| PDF-to-image (PyMuPDF 200 DPI) | 2 pages/PDF, 1653×2339 px, RGB |
| EasyOCR Thai recognition | Correctly extracts party names, candidate names, section labels |
| Form structure detection | ส.ส. 5/18 header, section labels, constituency number |

### Limitation Found
| Issue | Root Cause | Phase 2 Solution |
|-------|-----------|------------------|
| Vote counts = garbled (`.!.!..`) | Numbers are **handwritten** in the forms | Use Typhoon OCR API (Thai handwriting specialist) |
| Station number not in OCR text | Printed in a field that OCR skips (layout issue) | Extract from directory path (reliable alternative) |

### Next Steps (Phase 2)
1. Integrate Typhoon OCR API for handwritten digit recognition.
2. Combine directory-path metadata (station, district) with OCR candidate/party data.
3. Scale to all 60 polling stations and export `election_structured_data.csv`.